# 05 · Airport archetypes

**RQ3.** Do U.S. airports fall into distinct delay-risk archetypes once size and
cause mix are considered together?

Clustering is reported with its selection evidence — a silhouette sweep, a Ward
cross-check and a subsample stability score — because a k-means solution that
changes when a third of the airports are removed describes the sample, not the
airports.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from team01 import config, io, clean

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)
print(config.summarise())

Team01-FlightDelay-Project v0.2.0 | mode=sample | years=2015-2024 | flights=flights_sample.parquet


In [2]:
from team01 import cluster, features, viz

panel = clean.completed(io.load_flights())
airports = io.load_airports()

matrix = features.airport_feature_matrix(panel, airports, min_departures=2000)
print(f"{len(matrix)} airports x {matrix.shape[1]} columns")
matrix.head(3)

16 airports x 16 columns


,origin,n_dep,mean_arr_delay,p90_arr_delay,share_delayed_60,share_internal,share_late_aircraft,share_nas,share_weather,seasonal_amplitude,latitude_deg,longitude_deg,type,municipality,name,hub_stratum
0,ATL,4175,9.596340,53.756002,0.090778,0.673150,0.320403,0.198258,0.115162,9.049616,33.6407,-84.4277,large_airport,Atlanta,Atlanta International,3.0
1,AUS,2701,9.252466,54.848301,0.091448,0.673150,0.339150,0.201605,0.113050,16.052284,30.1975,-97.6664,medium_airport,Austin,Austin International,2.0
2,BNA,2889,9.796278,58.199684,0.096573,0.673682,0.328375,0.191046,0.123902,12.207685,36.1263,-86.6774,medium_airport,Nashville,Nashville International,2.0


In [3]:
sweep = cluster.sweep_k(matrix)
chosen = cluster.choose_k(sweep)
print(f"chosen k = {chosen}")
sweep

D:\Anaconda\Lib\site-packages\sklearn\cluster\_kmeans.py:1428: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
D:\Anaconda\Lib\site-packages\sklearn\cluster\_kmeans.py:1428: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
D:\Anaconda\Lib\site-packages\sklearn\cluster\_kmeans.py:1428: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
D:\Anaconda\Lib\site-packages\sklearn\cluster\_kmeans.py:1428: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than availabl

chosen k = 3


D:\Anaconda\Lib\site-packages\sklearn\cluster\_kmeans.py:1428: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


,k,silhouette,calinski_harabasz,davies_bouldin,inertia,smallest_cluster,largest_cluster
0,3,0.1706,4.2,1.5232,96.9,4,7
1,4,0.1678,4.1,1.3720,79.3,3,5
2,5,0.1835,4.3,1.0516,62.1,1,5
3,6,0.1516,4.0,0.9245,53.6,1,4
4,7,0.1373,3.8,0.8664,44.9,1,4
5,8,0.1158,3.7,0.7019,37.6,1,4


In [4]:
labelled, _, _ = cluster.fit(matrix, chosen)
names = cluster.name_clusters(labelled)
labelled["archetype"] = labelled["cluster"].map(names)

print(f"Ward cophenetic correlation {cluster.ward_check(matrix)['cophenetic_correlation']}")
print(cluster.stability(matrix, chosen))

cluster.cluster_table(labelled)[["cluster", "archetype", "airports",
                                 "mean_arr_delay", "share_delayed_60",
                                 "largest_members"]]

D:\Anaconda\Lib\site-packages\sklearn\cluster\_kmeans.py:1428: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
D:\Anaconda\Lib\site-packages\sklearn\cluster\_kmeans.py:1428: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
D:\Anaconda\Lib\site-packages\sklearn\cluster\_kmeans.py:1428: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


Ward cophenetic correlation 0.5793


D:\Anaconda\Lib\site-packages\sklearn\cluster\_kmeans.py:1428: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
D:\Anaconda\Lib\site-packages\sklearn\cluster\_kmeans.py:1428: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


D:\Anaconda\Lib\site-packages\sklearn\cluster\_kmeans.py:1428: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
D:\Anaconda\Lib\site-packages\sklearn\cluster\_kmeans.py:1428: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
D:\Anaconda\Lib\site-packages\sklearn\cluster\_kmeans.py:1428: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


D:\Anaconda\Lib\site-packages\sklearn\cluster\_kmeans.py:1428: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
D:\Anaconda\Lib\site-packages\sklearn\cluster\_kmeans.py:1428: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


{'mean_ari': 0.4869, 'min_ari': 0.2834, 'draws': 10, 'subsample_fraction': 0.8, 'verdict': 'unstable - report with caution'}


D:\Anaconda\Lib\site-packages\sklearn\cluster\_kmeans.py:1428: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
D:\Anaconda\Lib\site-packages\sklearn\cluster\_kmeans.py:1428: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


,cluster,archetype,airports,mean_arr_delay,share_delayed_60,largest_members
0,1,"Mid-size airport, mixed-cause (late-aircraft d...",7,10.752363,0.090973,"ATL, BNA, DTW, SEA, BTV, JFK"
1,2,"Mid-size airport, mixed-cause (strong seasonal...",5,11.126185,0.088484,"ORD, BOI, FSD, MHT, AUS"
2,0,"Mid-size airport, cascade-dominated",4,8.036485,0.083443,"DFW, DEN, LAX, MSP"


In [5]:
viz.fig_silhouette(sweep, chosen)
viz.fig_cluster_scatter(labelled, names)

[viz] 09_silhouette.png


[viz] 10_cluster_scatter.png


'C:\\Users\\lenovo\\Desktop\\Semester Project\\Team01-FlightDelay-Project\\figures\\10_cluster_scatter.png'